In [1]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# System Paths
RESULTS_DIR = Path("../../ofat_results_new")
OUTPUT_DIR = Path("ofat_analysis")

# Available Metrics:
# pct_satisfied
# voluntary_stays
# city_mean_income
# mean_neighbor_income
# mean_neighbor_income_variance
# mean_rent
# mean_utility
# mean_value
# mean_vision
# rent_income_timescale_ratio
# gini_coefficient
# homeless_fraction
# theil_index
# moran_i
# spatial_entropy
# neighborhood_heterogeneity
# income_mobility_indicator
# segregation_index
# gentrification_indicator

# Array of all target metrics tracked in your Mesa model datacollector CSVs
TARGET_METRICS = [
    "city_mean_income",
    "mean_neighbor_income",
    "mean_neighbor_income_variance",
    "mean_rent",
    "mean_utility",
    "mean_value",
    "gini_coefficient",
    "theil_index",
    "moran_i",
    "segregation_index",
    "homeless_fraction",
    "spatial_entropy",
    "neighborhood_heterogeneity",
    "income_mobility_indicator",
    "gentrification_indicator"
]

# --- Agent Analysis Dynamics Config ---
TARGET_PARAM_NAME = "neighborhood_radius"
TARGET_PARAM_VALUE = "3"

# Dynamically scan the folders matching this specific parameter scenario
AGENT_FILE_PATTERN = f"../../ofat_results/{TARGET_PARAM_NAME}/{TARGET_PARAM_VALUE}/run_*/agents_run_*.csv"

# Quantile binning settings
NUM_QUANTILES = 5
QUANTILE_LABELS = [f"Q{i+1}" for i in range(NUM_QUANTILES)]

# Visual Styling Configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 300

In [3]:
# Create the visual outputs target directory if it does not already exist
# Ensure explicit subdirectories exist inside your main output directory
(OUTPUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "parameters").mkdir(parents=True, exist_ok=True)

# Discover all generated model run CSV files
csv_files = sorted(list(RESULTS_DIR.glob("**/model_run_*.csv")))
total_files = len(csv_files)

print(f"Status: OK")
print(f"Data Directory: {RESULTS_DIR.resolve()}")
print(f"Target Output Folder: {OUTPUT_DIR.resolve()}")
print(f"Total Simulation Files Found: {total_files}")

Status: OK
Data Directory: /var/home/rick/Studies/Masters/ABM/ABM_Gentrification/ofat_results
Target Output Folder: /var/home/rick/Studies/Masters/ABM/ABM_Gentrification/src/experiments/ofat_analysis
Total Simulation Files Found: 650


In [4]:
raw_data = []

for file_path in csv_files:
    parts = file_path.parts
    param_name = parts[-4]
    param_raw_val = parts[-3]

    try:
        param_value = float(param_raw_val)
    except ValueError:
        param_value = param_raw_val

    try:
        df_run = pd.read_csv(file_path)
        if df_run.empty:
            continue

        # Isolate the final row row
        final_row = df_run.iloc[-1]

        # Build base mapping data payload
        row_payload = {
            "parameter": param_name,
            "value": param_value
        }

        # Inject every requested target metric column state
        for metric in TARGET_METRICS:
            if metric in df_run.columns:
                row_payload[metric] = final_row[metric]

        raw_data.append(row_payload)
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")

# Cast to Master DataFrame
df_master = pd.DataFrame(raw_data)
print(f"Master dataframe loaded with shape: {df_master.shape}")

Master dataframe loaded with shape: (650, 17)


In [5]:
print("--- DATAFRAME PROPERTIES ---")
print(f"Matrix Shape (Rows, Columns): {df_master.shape}\n")

print("--- NULL VALUE COUNT PER METRIC ---")
print(df_master[TARGET_METRICS].isnull().sum(), "\n")

print("--- HEAD PREVIEW ---")
df_master.head()

--- DATAFRAME PROPERTIES ---
Matrix Shape (Rows, Columns): (650, 17)

--- NULL VALUE COUNT PER METRIC ---
city_mean_income                 0
mean_neighbor_income             0
mean_neighbor_income_variance    0
mean_rent                        0
mean_utility                     0
mean_value                       0
gini_coefficient                 0
theil_index                      0
moran_i                          0
segregation_index                0
homeless_fraction                0
spatial_entropy                  0
neighborhood_heterogeneity       0
income_mobility_indicator        0
gentrification_indicator         0
dtype: int64 

--- HEAD PREVIEW ---


,parameter,value,city_mean_income,mean_neighbor_income,mean_neighbor_income_variance,mean_rent,mean_utility,mean_value,gini_coefficient,theil_index,moran_i,segregation_index,homeless_fraction,spatial_entropy,neighborhood_heterogeneity,income_mobility_indicator,gentrification_indicator
0,affordability_share,0.1,6.925407,7.242645,27.452568,0.720586,-0.712720,-0.932399,0.463104,0.408694,0.664355,0.292607,0.0,0.943305,0.392001,0.035398,0.105283
1,affordability_share,0.1,5.270355,5.457307,9.336194,0.544776,-0.496444,-0.492234,0.374441,0.254412,0.599866,0.372958,0.0,0.851360,0.344723,0.008696,0.079942
2,affordability_share,0.1,8.873775,9.211900,31.181575,0.900140,-0.861751,-0.908405,0.436537,0.340111,0.691634,0.322142,0.0,0.816816,0.392960,0.017391,0.133745
3,affordability_share,0.1,11.090223,11.780300,87.647051,1.193697,-0.998858,-1.166700,0.476164,0.434591,0.569468,0.293103,0.0,0.880331,0.448965,0.008621,0.170587
4,affordability_share,0.1,11.896967,12.022067,51.248246,1.200241,-1.165744,-0.987815,0.416951,0.308347,0.658673,0.423729,0.0,0.829341,0.400119,0.016949,0.178982


In [6]:
# Group by parameters and their assigned value steps
ofat_summary = (
    df_master.groupby(["parameter", "value"])[TARGET_METRICS]
    .agg(["mean", "std"])
)

# Display the summary table
ofat_summary.head(20)

city_mean_income            mean_neighbor_income  \
                                      mean        std                 mean   
parameter           value                                                    
affordability_share 0.1          10.134236   3.004923            10.689469   
                    0.2          13.823077   4.288649            14.320789   
                    0.3          18.754529   5.473793            18.996819   
                    0.4          21.189900   9.404658            21.590596   
                    0.5          19.610738   7.104618            19.642171   
                    0.6          26.840488  10.753796            26.831128   
                    0.7          23.898268   7.237247            23.885354   
                    0.8          33.709269  13.715342            33.398968   
                    0.9          40.235975  12.054348            40.046214   
                    1.0          33.761725   8.345015            33.352683   
discount_factor_max 0.2          27.785236   9.112000            27.619413   
                    0.4          32.141577   5.833782            31.906226   
                    0.6          30.355828  12.593599            30.151444   
                    0.8          25.904841  12.455622            25.702130   
                    1.0          25.543439   9.164637            25.323937   
                    1.2          26.227275   4.324254            25.993636   
                    1.4          24.121331   6.509344            23.988369   
                    1.6          26.472787  10.243005            26.441332   
                    1.8          33.984843  11.013107            33.676686   
                    2.0          36.621321  16.092072            36.462502   

                                     mean_neighbor_income_variance  \
                                 std                          mean   
parameter           value                                            
affordability_share 0.1     3.221933                    161.200448   
                    0.2     4.290831                    172.596811   
                    0.3     5.528706                    376.466356   
                    0.4     9.602673                    475.945869   
                    0.5     7.185795                    229.377554   
                    0.6    10.788386                    578.208146   
                    0.7     7.392913                    362.811474   
                    0.8    13.655468                   1522.612407   
                    0.9    12.079735                   1074.370151   
                    1.0     8.221046                    671.619822   
discount_factor_max 0.2     9.031495                    437.749184   
                    0.4     5.835376                    732.602334   
                    0.6    12.558220                    654.114835   
                    0.8    12.433869                    690.115291   
                    1.0     9.089400                    467.962783   
                    1.2     4.298601                    310.798947   
                    1.4     6.584223                    321.360557   
                    1.6    10.414376                    512.244212   
                    1.8    11.072477                   1062.507862   
                    2.0    16.079205                   3776.220089   

                                        mean_rent            mean_utility  \
                                   std       mean        std         mean   
parameter           value                                                   
affordability_share 0.1     360.969244   1.057515   0.312638    -1.202339   
                    0.2     146.832615   2.840419   0.848780     0.018974   
                    0.3     476.535765   5.577896   1.618119     2.200494   
                    0.4     513.825877   8.477355   3.781315     4.640508   
                    0.5     234.210479   9.634151   3.541819     6.498944   
    

In [7]:
unique_params = df_master["parameter"].unique()
num_plots = len(unique_params)

# Dynamic grid parameters (3 columns wide layout)
cols = 3
rows = (num_plots + cols - 1) // cols

# Loop through every defined metric to create its own isolated multi-panel file
for metric in TARGET_METRICS:
    if metric not in df_master.columns:
        print(f"Skipping visualization for '{metric}': column missing from extracted data context.")
        continue

    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows), sharey=False)
    axes = axes.flatten()

    for idx, param in enumerate(unique_params):
        ax = axes[idx]
        param_subset = df_master[df_master["parameter"] == param]

        # Lineplot automatically groups the 10 replicates per step into mean line + 95% CI band
        sns.lineplot(
            data=param_subset,
            x="value",
            y=metric,
            ax=ax,
            marker="o",
            color="#2b5c8f",
            errorbar="ci"
        )

        ax.set_title(f"Sensitivity to:\n{param}", fontsize=11, fontweight='bold')
        ax.set_xlabel("Parameter Step Value", fontsize=9)
        ax.set_ylabel(metric if idx % cols == 0 else "", fontsize=9)

    # Clean up any empty grids
    for empty_idx in range(idx + 1, len(axes)):
        fig.delaxes(axes[empty_idx])

    plt.tight_layout()

    # Construct structured file pathway output descriptor
    metric_image_path = OUTPUT_DIR / "metrics" / f"{metric}.png"
    plt.savefig(metric_image_path, dpi=300, bbox_inches='tight')
    plt.close() # Close plot instance to prevent heavy canvas memory leakages in loop execution

    print(f"Generated batch pipeline layout for: {metric_image_path}")

print("\nAll target model metrics processed successfully.")

Generated batch pipeline layout for: ofat_analysis/metrics/city_mean_income.png
Generated batch pipeline layout for: ofat_analysis/metrics/mean_neighbor_income.png
Generated batch pipeline layout for: ofat_analysis/metrics/mean_neighbor_income_variance.png
Generated batch pipeline layout for: ofat_analysis/metrics/mean_rent.png
Generated batch pipeline layout for: ofat_analysis/metrics/mean_utility.png
Generated batch pipeline layout for: ofat_analysis/metrics/mean_value.png
Generated batch pipeline layout for: ofat_analysis/metrics/gini_coefficient.png
Generated batch pipeline layout for: ofat_analysis/metrics/theil_index.png
Generated batch pipeline layout for: ofat_analysis/metrics/moran_i.png
Generated batch pipeline layout for: ofat_analysis/metrics/segregation_index.png
Generated batch pipeline layout for: ofat_analysis/metrics/homeless_fraction.png
Generated batch pipeline layout for: ofat_analysis/metrics/spatial_entropy.png
Generated batch pipeline layout for: ofat_analysis/me

In [16]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Define thematic groupings for the 15 remaining metrics
THEMATIC_GROUPS = {
    "Socio-Economic & Inequality": [
        "city_mean_income",
        "mean_neighbor_income",
        "mean_neighbor_income_variance",
        "gini_coefficient",
        "theil_index"
    ],
    "Housing & Real Estate Market": [
        "mean_rent",
        "mean_value"
    ],
    "Urban Spatial Segregation": [
        "moran_i",
        "segregation_index",
        "spatial_entropy",
        "neighborhood_heterogeneity"
    ],
    "Displacement & Community Well-being": [
        "mean_utility",
        "homeless_fraction",
        "income_mobility_indicator",
        "gentrification_indicator"
    ]
}

# Ensure destination directory exists
PARAMS_OUT_DIR = OUTPUT_DIR / "parameters"
PARAMS_OUT_DIR.mkdir(parents=True, exist_ok=True)

unique_params = df_master["parameter"].unique()
palette = sns.color_palette("tab10")

print(f"Compiling locally normalized thematic profiles for {len(unique_params)} parameters...")

# Outer loop isolates one system parameter at a time
for param in unique_params:
    # Extract raw data strictly for the current parameter
    param_subset = df_master[df_master["parameter"] == param].copy()

    if param_subset.empty:
        continue

    # --- LOCAL MIN-MAX NORMALIZATION ---
    # Normalize each metric based EXCLUSIVELY on its range within this parameter's dataset
    for theme_name, metrics in THEMATIC_GROUPS.items():
        for metric in metrics:
            if metric in param_subset.columns:
                local_min = param_subset[metric].min()
                local_max = param_subset[metric].max()

                # Prevent division by zero if a metric remains perfectly flat
                if local_max - local_min > 1e-8:
                    param_subset[metric] = (param_subset[metric] - local_min) / (local_max - local_min)
                else:
                    param_subset[metric] = 0.0  # Keep completely flat lines at baseline floor

    # Initialize a clean 2x2 multi-panel layout
    fig, axes = plt.subplots(2, 2, figsize=(15, 11), sharex=True)
    axes = axes.flatten()

    # Inner loop processes each distinct theme panel
    for idx, (theme_name, metrics) in enumerate(THEMATIC_GROUPS.items()):
        ax = axes[idx]
        color_idx = 0

        for metric in metrics:
            if metric not in param_subset.columns:
                continue

            clean_label = metric.replace('_', ' ').title()

            # Lineplot maps the locally normalized mean lines + 95% Confidence Band
            sns.lineplot(
                data=param_subset,
                x="value",
                y=metric,
                ax=ax,
                label=clean_label,
                color=palette[color_idx % len(palette)],
                linewidth=2,
                marker="o",
                markersize=5,
                errorbar="ci"
            )
            color_idx += 1

        # Panel Layout Enhancements
        ax.set_title(theme_name, fontsize=12, fontweight='bold', pad=10)
        ax.set_xlabel(f"Parameter Step Value ({param})", fontsize=10)
        ax.set_ylabel("Local Normalized Scale (0.0 - 1.0)", fontsize=10)

        # Maintain your clean uniform 0 to 1 limits
        ax.set_ylim(-0.05, 1.05)

        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend(loc="upper left", fontsize=9, framealpha=0.9)

    plt.suptitle(f"System Response Matrices: Relative Sensitivity to {param.upper()}", fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout()

    # Write file out to disk
    param_image_path = PARAMS_OUT_DIR / f"{param}.png"
    plt.savefig(param_image_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Successfully generated cohesive thematic profile: {param_image_path}")

print("\nAll relative parameter visualizations have been written to disk.")

Compiling locally normalized thematic profiles for 7 parameters...
Successfully generated cohesive thematic profile: ofat_analysis/parameters/affordability_share.png
Successfully generated cohesive thematic profile: ofat_analysis/parameters/discount_factor_max.png
Successfully generated cohesive thematic profile: ofat_analysis/parameters/income_similarity_min.png
Successfully generated cohesive thematic profile: ofat_analysis/parameters/initial_income_max.png
Successfully generated cohesive thematic profile: ofat_analysis/parameters/neighborhood_radius.png
Successfully generated cohesive thematic profile: ofat_analysis/parameters/rationality_max.png
Successfully generated cohesive thematic profile: ofat_analysis/parameters/risk_aversion_max.png

All relative parameter visualizations have been written to disk.
